# Exploration and Technical Demonstration of Databricks

In this notebook we wil be taking a look into:
1. Ingestion of NYC Taxi dataset into databricks
2. Process the raw data through the typical processing methodology from bronze -> silver -> gold
3. Show how processed data can be moved into the lakehouse
4. Demonstrate distributed workflows using pyspark workflows
5. Demonstrate pandas api on Spark.
6. Demonstrate the parallellism of Databricks/Spark.
7. Explore various data persistence mechanism on Databricks
8. Explore Delta Lake features.


### 1. Import Libraries

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, year, month, dayofmonth, hour, avg, count

We will be processing data through these steps:
1. Bronze Layer: Raw unprocessed data
2. Silver Layer: Cleaned and Standardized data (cast datatypes, handle nulls, filter bad rows)
3. Gold: Aggregated/curated (e.g., average fare per borough, trip counts per hour)

### 2. Ingest Data (Bronze Layer)

- Spark doesn’t read the CSV file line by line like pandas.
- It splits the file into partitions (chunks of rows), and each partition is processed in parallel across the cluster’s worker nodes.
- If you upload multiple monthly taxi files, Spark will distribute them across executors automatically.


Ingest data from S3

In [0]:
s3path = 's3://nyctaxidatabricks/yellow_tripdata_2025-01.parquet'
df_bronze = (spark.read.parquet(s3path))
display(df_bronze.limit(5))  # Show sample rows
print("Bronze count:", df_bronze.count())

In [0]:
bronze_path = "/Volumes/workspace/default/nyc_taxi/yellow_tripdata_2025-01.parquet"

# Read raw parquet into Spark DataFrame
df_bronze = (spark.read.parquet(bronze_path))
display(df_bronze.limit(5))  # Show sample rows
print("Bronze count:", df_bronze.count())


### 3. Clean & Transform Data (Silver Layer)

- Each transformation (withColumn, filter, cast) is applied independently on each partition.
- Spark builds a logical execution plan (DAG) and executes tasks in parallel when you trigger an action (like .count() or .show()).

In [0]:
df_silver = (df_bronze
             .withColumn("pickup_datetime", col("tpep_pickup_datetime").cast("timestamp"))
             .withColumn("dropoff_datetime", col("tpep_dropoff_datetime").cast("timestamp"))
             .withColumn("passenger_count", col("passenger_count").cast("integer"))
             .withColumn("trip_distance", col("trip_distance").cast("double"))
             .withColumn("fare_amount", col("fare_amount").cast("double"))
             .filter(col("passenger_count") > 0)
             .filter(col("fare_amount") > 0))

display(df_silver.limit(5))
print("Silver count:", df_silver.count())

### 4. Curated Aggregations (Gold Layer)

- The groupBy operation is a shuffle: Spark redistributes rows across executors so that all rows for the same pickup_hour end up together.
- Each executor then computes partial aggregates in parallel, and Spark merges them into the final result.
- This is a textbook example of parallel distributed aggregation.


In [0]:
df_gold = (df_silver
           .withColumn("pickup_hour", hour(col("pickup_datetime")))
           .groupBy("pickup_hour")
           .agg(avg("fare_amount").alias("avg_fare"),
                count("*").alias("trip_count"))
           .orderBy("pickup_hour"))

display(df_gold)

### 5. Pandas API on Spark Demo

- Even though the syntax looks like pandas, Spark executes the operations in parallel across the cluster.
- For example, describe() computes statistics (mean, std, min, max) by scanning partitions in parallel and combining results.


In [0]:
import pyspark.pandas as ps

# Convert Spark DataFrame to pandas-on-Spark
psdf = df_silver.pandas_api()

# Pandas-style operations (executed on Spark cluster)
summary = psdf[["fare_amount", "trip_distance"]].describe()
display(summary)

# GroupBy example
avg_fares = psdf.groupby("passenger_count")["fare_amount"].mean()
display(avg_fares)

### 6. Persist Data in Delta Lake

- Writing is also parallelized: each partition is written out as a separate Parquet/Delta file.
- This makes reads and writes scalable — multiple executors handle chunks of the dataset simultaneously.

In [0]:
# Save as a Delta table in the default catalog/schema
df_silver.write.format("delta").mode("overwrite").saveAsTable("nyc_taxi_silver")

df_gold.write.format("delta").mode("overwrite").saveAsTable("nyc_taxi_gold")

#### Demonstrating Spark SQL

In [0]:
%sql
SELECT * FROM nyc_taxi_silver LIMIT 10;

In [0]:
%sql

SELECT * FROM nyc_taxi_gold LIMIT 10;

### 7. Export Data to External Storage

In [0]:
export_path = "/Volumes/workspace/default/nyc_taxi/nyc_taxi_gold_export"
df_gold.write.mode("overwrite").parquet(export_path)